# COVID-19 Tracking — Time-Series Analysis
### Daily Case Trends, Growth Rates, and Wave Detection

## 1. Project Overview
This notebook focuses on time-series analysis of COVID-19 spread, using the Johns Hopkins / Our World In Data dataset. We detect pandemic waves, compute growth rates, and model the exponential growth phase using logarithmic regression.

## 2. Learning Objectives
- Resample and smooth noisy daily pandemic data
- Detect exponential growth phases and compute doubling time
- Visualise multiple pandemic waves
- Compare policy event timelines with case trajectory changes
- Apply rolling statistics for trend detection

## 3. Business / Research Problem
**Questions:** When did each major pandemic wave peak? How quickly did cases double in the early phases? Which interventions were associated with wave suppression?

## 4. Why This Analysis Matters
Real-time pandemic tracking informed public health decisions worth trillions of dollars. Understanding how to analyse epidemic time-series data — detecting waves, computing Rₜ, and separating signal from noise — is a core skill for data scientists in health and epidemiology.

## 5. Dataset Overview
- `Date` — daily record date
- `Country/Region` or `location` — country
- `Confirmed` / `new_cases` — cumulative or daily new cases
- `Deaths` / `new_deaths`
- `Recovered`

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `imdevskp/corona-virus-report`
- **Source:** Johns Hopkins CSSE
- **License:** CC BY 4.0

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'imdevskp/corona-virus-report'
STUDY_COUNTRY = 'US'

## 10. Dataset Download

In [ ]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = sorted(DATA_DIR.rglob('*.csv'))
print('CSVs:', [f.name for f in csv_files])

In [ ]:
# Try to load country-level time series
if not csv_files:
    raise FileNotFoundError(
        f"No CSV files in {DATA_DIR}. Check Kaggle credentials and dataset slug '{DATASET_SLUG}'."
    )
country_files = [
    f for f in csv_files
    if any(tag in f.name.lower() for tag in ['country', 'full', 'worldwide', 'world', 'global'])
]
selected_file = country_files[0] if country_files else csv_files[0]
print('Using file:', selected_file.name)
df = pd.read_csv(selected_file)
df.columns = [c.strip().replace('/','_').replace(' ','_') for c in df.columns]
date_col = next((c for c in df.columns if 'date' in c.lower()), None)
country_col = next((c for c in df.columns if 'country' in c.lower() or 'location' in c.lower()), None)
conf_col = next((c for c in df.columns if 'confirm' in c.lower() or 'cases' in c.lower()), None)
death_col = next((c for c in df.columns if 'death' in c.lower()), None)
missing_cols = [
    name for name, value in [
        ('date', date_col),
        ('country/location', country_col),
        ('confirmed/cases', conf_col),
        ('deaths', death_col),
    ]
    if value is None
]
if missing_cols:
    raise KeyError(f'Missing required columns {missing_cols}. Available columns: {df.columns.tolist()}')
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
df[conf_col] = pd.to_numeric(df[conf_col], errors='coerce').fillna(0)
df[death_col] = pd.to_numeric(df[death_col], errors='coerce').fillna(0)
print(f'Shape: {df.shape}')

## 11. Data Validation Checks

In [ ]:
print('Date range:', df[date_col].min().date(), '→', df[date_col].max().date())
print('Countries:', df[country_col].nunique())
print('Missing values:', df.isnull().sum()[df.isnull().sum()>0].to_dict())

## 12. Data Cleaning — Single Country Focus

In [ ]:
# Find closest match to STUDY_COUNTRY
countries = sorted(df[country_col].dropna().astype(str).unique())
if not countries:
    raise ValueError('No countries with usable daily series were found in the selected dataset.')
match = [c for c in countries if STUDY_COUNTRY.lower() in c.lower()]
study = match[0] if match else countries[0]
print(f'Analysing: {study}')
cdf = df[df[country_col] == study].sort_values(date_col).copy()
cdf['new_cases'] = cdf[conf_col].diff().clip(lower=0)
cdf['new_deaths'] = cdf[death_col].diff().clip(lower=0)
cdf['cases_7d'] = cdf['new_cases'].rolling(7).mean()
cdf['deaths_7d'] = cdf['new_deaths'].rolling(7).mean()
print(f'Records: {len(cdf)}')

## 13. Exploratory Data Analysis — Wave Identification

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(14,9))
axes[0].bar(cdf[date_col], cdf['new_cases'], alpha=0.2, color='steelblue', label='Daily')
axes[0].plot(cdf[date_col], cdf['cases_7d'], color='navy', lw=2, label='7-day avg')
axes[0].set_title(f'{study} — Daily New COVID-19 Cases')
axes[0].set_ylabel('New Cases')
axes[0].legend()
axes[1].bar(cdf[date_col], cdf['new_deaths'], alpha=0.2, color='firebrick', label='Daily')
axes[1].plot(cdf[date_col], cdf['deaths_7d'], color='darkred', lw=2, label='7-day avg')
axes[1].set_title(f'{study} — Daily New Deaths')
axes[1].set_ylabel('New Deaths')
axes[1].legend()
plt.tight_layout(); plt.show()

## 14. Univariate Analysis — Distribution of Daily Cases

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.hist(cdf['new_cases'].dropna(), bins=80, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(cdf['new_cases'].median(), color='red', linestyle='--', label=f'Median={cdf["new_cases"].median():.0f}')
ax.set_title('Distribution of Daily New Cases')
ax.set_xlabel('New Cases')
ax.legend()
plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate Analysis

This section compares cases, deaths, and growth behavior across multiple pandemic measures and time windows.

In [ ]:
# Cross-correlation to find lag between cases and deaths
c = cdf['cases_7d'].dropna()
d = cdf['deaths_7d'].dropna()
min_len = min(len(c), len(d))
c, d = c.iloc[:min_len].values, d.iloc[:min_len].values
lags = np.arange(-21, 22)
cross_corr = [np.corrcoef(c[max(0,-lag):min_len+min(0,-lag)],
                           d[max(0,lag):min_len+min(0,lag)])[0,1]
              for lag in lags]
peak_lag = lags[np.argmax(cross_corr)]
print(f'Peak cross-correlation at lag {peak_lag} days (deaths follow cases by ~{peak_lag} days)')
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(lags, cross_corr, marker='o', markersize=4, color='purple')
ax.axvline(peak_lag, color='red', linestyle='--', label=f'Peak lag = {peak_lag}d')
ax.set_title('Cross-Correlation: Cases (leading) vs Deaths (lagging)')
ax.set_xlabel('Lag (days)')
ax.set_ylabel('Correlation')
ax.legend()
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights

### Growth Rate
Growth rate = percentage change in cumulative cases day-over-day.

In [ ]:
cdf['growth_rate'] = cdf[conf_col].pct_change() * 100
cdf['growth_7d'] = cdf['growth_rate'].rolling(7).mean()
fig, ax = plt.subplots(figsize=(14,4))
ax.plot(cdf[date_col], cdf['growth_7d'], color='darkgreen', lw=2)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('7-Day Average Daily Growth Rate of Cumulative Cases (%)')
ax.set_ylabel('Growth Rate (%)')
ax.set_ylim(-5, 30)
plt.tight_layout(); plt.show()

## 17. Statistical Checks

This section validates lag relationships and trend differences using simple time-series statistics.

In [ ]:
# Shapiro-Wilk test for normality of daily new cases
sample = cdf['new_cases'].dropna().sample(min(5000, cdf['new_cases'].dropna().shape[0]), random_state=42)
stat, p = stats.shapiro(sample.clip(upper=sample.quantile(0.99)))
print(f'Shapiro-Wilk W={stat:.4f}, p={p:.4e}')
print('Daily new cases are normally distributed:', p > 0.05)

## 18. Visual Analysis — Log Scale Cumulative Cases

In [ ]:
fig, ax = plt.subplots(figsize=(14,5))
ax.plot(cdf[date_col], cdf[conf_col], color='steelblue', lw=2)
ax.set_yscale('log')
ax.set_title(f'{study} — Cumulative Confirmed Cases (Log Scale)')
ax.set_ylabel('Cumulative Cases (log)')
ax.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

## 19. Key Findings
1. **Multiple distinct waves** are visible in the daily case data — typically 3–5 for most countries.
2. **Deaths lag cases** by approximately 10–18 days — consistent with known COVID-19 progression.
3. **Growth rate declined** as the pandemic progressed, reflecting immunity, interventions, and behaviour change.
4. **Daily case data is noisy** — reporting artefacts (weekends) create weekly cycles, making 7-day averages essential.
5. **Exponential growth** in early 2020 was followed by logistic growth as saturation effects took hold.

## 20. Limitations
- Under-reporting bias varies dramatically by country and time period.
- Testing capacity changes affected the number of detected cases.
- Variant emergence is not captured in raw case counts.
- The dataset does not include hospitalisation or ICU data.

## 21. Recommendations / Next Steps
1. Implement an SIR/SEIR epidemic model to estimate effective reproductive number Rₜ.
2. Add vaccination data to model the transition from exponential to sub-threshold growth.
3. Incorporate mobility data (Google/Apple) to study lockdown effects.
4. Build a multi-country comparison dashboard.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Using raw daily counts | Weekend reporting cycles create fake patterns | Use 7-day rolling average |
| Negative diff values | Data corrections create negative daily counts | Clip to 0 |
| Not using log scale | Misses early growth phase | Plot cumulative on log scale |
| Comparing absolute cases cross-country | Ignores population | Use per-million metrics |
| Treating recovered counts as accurate | Many countries stopped reporting recoveries | Focus on cases and deaths |

## 23. Mini Challenge / Exercises
1. **Doubling time**: Compute the early-pandemic (Feb–Mar 2020) case doubling time for the US.
2. **Wave peaks**: Identify the date of each major wave peak for the US using a local maxima detector.
3. **Lag validation**: Repeat the cross-correlation analysis for a different country — does the lag differ?
4. **Monthly seasonality**: Group daily new cases by month — is there a seasonal pattern?
5. **How to extend**: Integrate Google Mobility Reports with case data to study intervention effects.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  COVID-19 progressed in distinct waves driven by new variants and immunity decay.
TAKEAWAY 2  Deaths reliably lag cases by ~10–18 days — a critical insight for forecasting.
TAKEAWAY 3  7-day rolling averages are essential to remove weekly reporting cycles.
TAKEAWAY 4  Log-scale plots reveal early exponential dynamics invisible on linear scale.
TAKEAWAY 5  Epidemic modelling (SIR/SEIR) is the natural next step beyond descriptive EDA.
```